# Semantic Chunking

Using embedding similarities between sentences to decide chunk boundaries to ensure every chunk is coherent and no data is cut off mid sentence.


In [ ]:
import json
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

# Langchain Specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()


In [1]:
# from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

d:\Learning\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Learning\RAG\.venv\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.26.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
## Initialize a simple Embedding model (no API Key needed!)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model_name="sentence-transformers/paraphrase-multilingual-MinLM-L12-v2",
    multi_process=True,
    show_progress=True,
    cache_folder="./../model_cache/",
    # model_kwargs = {"device": "cpu"}
    # model_kwargs = {"device": "gpu"}
    # model_kwargs = {"device": "cuda"}
)
embedding_model



HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder='./../model_cache/', model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=True, show_progress=True)

In [3]:
# Step1: load and split sentences.